In [1]:
!pip -q install duckdb pyarrow
from google.colab import files
up = files.upload()
PARQUET = list(up.keys())[0]

Saving df_merge_clean.parquet to df_merge_clean (1).parquet


In [11]:
#!pip -q install duckdb pyarrow
import duckdb
#from google.colab import files
#up = files.upload()  # envie df_merge_clean.parquet
#PARQUET = list(up.keys())[0]

sql = f"""
WITH base AS (
  SELECT CAST(pdv AS BIGINT) pdv,
         CAST(produto AS BIGINT) produto,
         iso_week, iso_year,
         CAST(quantity AS DOUBLE) qty
  FROM read_parquet('{PARQUET}')
  WHERE iso_year = 2022
),
recent AS (
  SELECT pdv, produto,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS qty_recent
  FROM base GROUP BY 1,2
),
pairs AS (
  SELECT pdv, produto
  FROM recent
  WHERE qty_recent > 0
  ORDER BY qty_recent DESC
  LIMIT 250000           -- <<< CAP_PARES
),
seed AS (
  SELECT b.pdv, b.produto,
         AVG(qty) FILTER (WHERE iso_week BETWEEN 49 AND 52) AS mean4,
         SUM(qty) FILTER (WHERE iso_week BETWEEN 45 AND 52) AS sum8
  FROM base b
  JOIN pairs p USING(pdv,produto)
  GROUP BY 1,2
),
seed_final AS (
  SELECT pdv, produto,
         COALESCE(mean4, sum8/8.0, 0.1) AS qty_seed
  FROM seed
),
weeks AS (SELECT * FROM (VALUES (1),(2),(3),(4),(5)) AS t(semana))
SELECT w.semana,
       CAST(s.pdv AS BIGINT)     AS pdv,
       CAST(s.produto AS BIGINT) AS produto,
       CAST(s.qty_seed AS DOUBLE) AS quantidade
FROM weeks w CROSS JOIN seed_final s
"""
con = duckdb.connect()
sub = con.execute(sql).df()
print("rows:", len(sub))
OUT = "submission.parquet"
con.execute(f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')")
files.download(OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 1250000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
OUT = "submission.parquet"
con.execute(f"COPY ({sql}) TO '{OUT}' (FORMAT 'parquet')")
from google.colab import files; files.download(OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import pyarrow.parquet as pq, pandas as pd
df = pq.read_table("submission.parquet").to_pandas()
assert list(df.columns) == ['semana','pdv','produto','quantidade']
assert set(df['semana']) <= {1,2,3,4,5}
assert df['quantidade'].ge(0).all()
assert df.duplicated(['semana','pdv','produto']).sum() == 0
print("ok:", df.shape)

ok: (1250000, 4)


In [15]:
OUT = "submission_int.parquet"
con.execute(f"""
COPY (
  SELECT
    CAST(semana  AS INT)    AS semana,
    CAST(pdv     AS BIGINT) AS pdv,
    CAST(produto AS BIGINT) AS produto,
    CAST(GREATEST(0, ROUND(quantidade)) AS BIGINT) AS quantidade
  FROM ({sql})
) TO '{OUT}' (FORMAT 'parquet')
""")
from google.colab import files; files.download(OUT)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import pyarrow.parquet as pq
print(pq.read_table(OUT).schema)
# semana: int32, pdv: int64, produto: int64, quantidade: int64

semana: int32
pdv: int64
produto: int64
quantidade: int64
